# Setup

Member D — controlled CLIP improvements. This is a **new notebook**; `approach3_clip.ipynb` and its outputs are reference artifacts. All source changes are isolated in new helpers. Run on `member-d`.

| Experiment | Change | Retrain? |
|---|---|---|
| E1 | WiSE-FT: `(1-alpha)*original + alpha*Partial-6`, in state-dict space | No |
| E2 | Prompt ensemble on existing Partial-6; healthy gets healthy prompts | No |
| E3 | Partial-6 with CE + lambda × original-image-feature preservation | Yes |
| E4 | All visual parameters trainable; text and logit_scale frozen | Yes |
| E5 | E4 + preservation + ensemble, followed by validation-selected WiSE-FT | Yes |

**Fairness:** preserve the original shared split (7,223 train / 777 seen validation), seed 42, augmentation, AdamW, LR 1e-5, weight decay .05, batch 8, 10 epochs and float32 for full runs. Original Partial-6 used exactly these settings. No scheduler, gradient clipping or extra augmentation is added. All retrained candidates start from the same original OpenAI weights, not the previously trained Partial-6. E1/E2 explicitly reuse existing Partial-6.

E1/E2 need a formal Partial-6 `.pt`. Set `CLIP_PARTIAL6` if it is not in project checkpoints; the inspected Downloads copy is also detected. Optional `CLIP_BASELINE_RESULTS` points to the original comparison JSON. Baseline Full needs its checkpoint next to Partial-6 to rerun it. If unavailable, report its reason and optionally retain historical results separately.

DEBUG uses batch 1, 2 images per shared DEBUG split, 1 epoch, a 3-point alpha grid and one positive lambda. Every experiment still runs. These tiny results cannot select scientific improvements. Full mode never silently shrinks its workload. A frozen teacher on CPU, cached text embeddings, small text chunks and gradient checkpointing limit GPU memory; actual RTX 4050/5090 execution must be verified on those devices.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'common/dataset.py').is_file())
sys.path.insert(0, str(ROOT))
from IPython.display import display
from approach3_clip_improvements.approaches.clip_improvements import ImprovementRun, default_hp
from approach3_clip_improvements.approaches.clip_improvement_core import check_data, file_hash
DEBUG = True
HP = default_hp()
PARTIAL6_PATH = os.getenv('CLIP_PARTIAL6')  # Auto-detect project checkpoints or the inspected Downloads copy.
BASELINE_RESULTS = os.getenv('CLIP_BASELINE_RESULTS', str(ROOT / 'approach3_clip_improvements/reference_results/approach3_clip_comparison.json'))
if BASELINE_RESULTS is None and (Path.home() / 'Downloads/approach3_clip_comparison.json').is_file():
    BASELINE_RESULTS = str(Path.home() / 'Downloads/approach3_clip_comparison.json')
print('DEBUG:', DEBUG, 'requested full-run settings:', HP)

# Data

Reuse `common.config`, `get_loaders(batch_size, norm='clip', debug=...)`, `get_pairs()` and the existing `evaluate()` API. The checked repository uses seven CSV columns, integer `(crop,disease)` pairs, and `pair_row` indexes: 0–322 seen, 323–332 unseen. The real name mapping contains the disease label `healthy`; no invented class mapping is used.

**Use the old validation setup. No extra combinations are removed from training.** Per your decision, no proxy-unseen split is introduced. Shared per-pair sampling yields 777 validation images (rounding per pair), not exactly 800. Every candidate gets the same membership fingerprints, seed, batch order and augmentation protocol. Worker initialization explicitly seeds NumPy/Python. Evaluation uses deterministic zero-worker iteration with identical membership, normalization and batch size; this runtime choice is logged.

The shared pair table reads official CSV metadata for candidate labels/names, which is permitted side information. Test image pixels and test accuracy are not used for choosing alpha, lambda, prompts or epochs. Image-name overlap, counts and label ranges are checked. Seen validation cannot directly establish unseen generalization. Original official test results have already been viewed; this follow-up is exploratory, and that history must be disclosed in the report.

In [ ]:
pairs = check_data()
print('Candidate tensor:', tuple(pairs.shape))
original_hash = file_hash(ROOT / 'approaches/approach3_clip.ipynb')

# Model

OpenCLIP ViT-B/16 with OpenAI pretrained weights and QuickGELU matches the original notebook. Original CLIP is the frozen CPU teacher. Strict checkpoint loading checks keys, shapes, dtypes, finite weights, all 333 pair rows, prompt order and formal-run metadata. The existing Partial-6 text and logit weights must match original CLIP before its frozen text cache can be reused.

Scores are `exp(logit_scale) * normalized_image_features @ normalized_text_features.T`. WiSE-FT interpolates the stored parameters, including the **logarithmic** logit_scale parameter, never its exponent; non-floating buffers are copied from the original base state. Text ensembles normalize each template embedding, average within each candidate, then normalize again. Existing baselines retain the original single prompt for a clear ablation.

Text remains frozen in E3/E4/E5. Partial-6 unfreezes only the last six visual blocks. E4/E5 unfreeze the entire visual encoder including its output projection, with global logit_scale frozen. These exact parameter choices and full counts are logged. Unexpected architectures or incompatible checkpoints cause explicit errors. No model state is silently loaded with `strict=False`.

In [ ]:
run = ImprovementRun(HP, debug=DEBUG, partial_path=PARTIAL6_PATH, baseline_results=BASELINE_RESULTS)
print('Strict compatibility:', run.compatibility)
print('Run:', run.run_id, 'samples:', run.counts, 'device:', run.device)

# Training

All five stages perform **selection on validation only**. E1 and E5 use the predefined alpha grid; ties favour smaller alpha. E2 compares predefined 3/5-template ensembles; ties favour fewer templates. E3 compares positive preservation lambdas; ties favour smaller lambda. Every training trial selects its earliest maximum-validation-Top-1 epoch over all 333 candidates.

E5 transfers E2's prompt winner and E3's lambda winner **unchanged**, trains a fresh Visual-Full student from original CLIP, then selects alpha on that student's validation scores. This is a predeclared staged selection procedure, not independent confirmation on another validation set. A successful component is not assumed to make the combination better. There is no official-test evaluation until all five stages are sealed.

The preservation teacher processes the exact same augmented image under `no_grad`; teacher/student dimensions and frozen gradients are checked. CE remains over 323 seen pairs. Every trial resets seed, original weights and loaders. Logs include CE, preservation and total training loss, validation loss/Top-1, epoch time, best epoch and trainable/frozen parameter names/counts. Checkpoints have strict disk save/reload checks. DEBUG removes its new temporary checkpoint files after checking and retains states in CPU memory for evaluation; full mode permanently retains all selected weights.

Full grids increase validation search compared with the original baseline; report that extra tuning budget. No best outcome is guaranteed and no significance claim is justified by one seed.

## E1 — WiSE-FT, no retraining

In [ ]:
e1 = run.execute('wise_ft')

## E2 — Prompt ensemble, no retraining

Healthy candidates use two healthy-only descriptions. Diseased candidates use the selected 3/5 descriptions from HP. All candidate rows remain in shared order.

In [ ]:
e2 = run.execute('prompt_ensemble')

## E3 — Partial-6 with feature preservation

In [ ]:
e3 = run.execute('preservation')

## E4 — Visual-Full, Text-Frozen

In [ ]:
e4 = run.execute('visual_full')

## E5 — Combined hypothesis

In [ ]:
e5 = run.execute('combined')
frozen_selection = run.execute('seal_selection')
display(frozen_selection)

# Evaluation

Selection has now been frozen in `selection_frozen.json`. The evaluation stage refuses altered settings or a second evaluation into existing outputs. It reruns shared evaluation for original Zero-shot, existing Partial-6, optional existing Full and all five selected experiments. Each model has its own JSON and confusion PNG/CSV under a unique run directory. Crop/disease accuracy and simulated brightness/blur/noise robustness are seen-test measures, matching common evaluation.

The Full baseline must re-encode text from its own final model state; it cannot reuse original frozen text embeddings. Missing Full checkpoints are explicitly marked unavailable. Historical Full metrics, if provided, are retained as historical records; they are not inserted into tiny DEBUG comparison rows.

A full test pass does not make this a fresh untouched benchmark: prior baseline test results motivated this exploratory follow-up. Do not choose another hyperparameter after viewing these test results.

In [ ]:
comparison = run.execute('evaluate_all')
display(comparison.round(2))
full_status = run.reports['baseline_full']
if full_status['status'] != 'evaluated':
    print('Full baseline unavailable:', full_status['reason'])
    if 'historical_metrics' in full_status:
        print('Historical Full metrics are retained in comparison.json, separately from this run.')

# Save

The automatically generated comparison table is also exported to CSV. Machine-readable comparison JSON contains environment, exact source-checkpoint hash, split fingerprints, candidate selection histories, effective DEBUG settings, official metrics and artifact paths. Full-mode trained weights and intermediate E5 weights are under a **new** `checkpoints/clip_improvements/<run_id>/`; old notebooks, checkpoints and result files are never overwritten. Candidate grids and current best-epoch states are kept in memory during selection; only selected models are permanently saved in full mode. DEBUG checkpoint paths are ephemeral and explicitly labelled; its comparison JSON/CSV and confusion outputs remain available.

For school runs: copy the actual Partial-6 and optional Full checkpoints into `checkpoints/approach3_clip/`, install pinned requirements, pass DEBUG on the target GPU, then set DEBUG=False and restart/run all. The helper raises a configuration error on OOM instead of secretly changing batch size or skipping experiments. Leave enough disk space for several FP32 checkpoints (several GB). Back up the new checkpoint/result directories before logging off; Git does not back up weights.

Limitations: same seen-only validation, one seed, additional selection budget, optional unavailable Full checkpoint, DEBUG's tiny subsets, GPU compatibility still needing target-hardware checks, and simulated rather than independent real-field robustness. A final comparison can show that E5 fails to improve; retain that result.

In [ ]:
table_path = run.save_table()
assert file_hash(ROOT / 'approaches/approach3_clip.ipynb') == original_hash
print('Table:', table_path)
print('JSON / confusion outputs:', run.result_dir)
print('New checkpoints:', run.checkpoint_dir)
run.close()